# **FullGrad, NoiseGrad and FusionGrad: network biases and noise in the weights**

A practice for the module ["Heuristic gradient-based explanations"](https://ai-interpretability.school).

The lesson says three things about three methods, and each of them can be checked with a number:

1. for a network built from ReLU and batch normalization the output decomposes **exactly** into the gradient times the input plus the contributions of all biases, and the bias contribution in such a network is "not small" (FullGrad, Srinivas and Fleuret, 2019);
2. the FullGrad map does not keep this decomposition: the absolute value of every term is taken before the summation, so the sum of the map no longer equals the output;
3. NoiseGrad averages maps over copies of the model with multiplicative noise in the weights, FusionGrad adds input noise on top of that, and the authors pick the weight-noise level so that the model's accuracy drops by about 5 % (Bykov et al., 2022).

The model and the picture are the same as in the notebook `PRACTICE_smoothgrad_sanity`: ResNet-50 with ImageNet weights and the photo of a wild boar from the course repository. The notebook runs on a CPU in a few minutes; FullGrad comes from the `grad-cam` library, NoiseGrad and FusionGrad we write ourselves, following the formulas of the lesson.

In [ ]:
!pip install -q grad-cam==1.5.7

In [ ]:
import copy
import time
import urllib.request
from io import BytesIO

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import spearmanr
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)   # all noise below comes from its own generators with a fixed seed
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval()
for m in model.modules():          # ReLU without in-place writes: otherwise it overwrites the batch-norm output
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load(name):
    return transform(Image.open(BytesIO(urllib.request.urlopen(DATA + name, timeout=30).read())).convert('RGB')).unsqueeze(0)


x = load('hog.jpg')
categories = urllib.request.urlopen(DATA + 'imagenet_classes.txt', timeout=30).read().decode().splitlines()
with torch.no_grad():
    logits = model(x)
target = int(logits.argmax())
print('class:', target, categories[target], '  logit:', round(logits[0, target].item(), 4))

**One change to the model.** ReLU in `torchvision` writes its result over its input (`inplace=True`), and the input of a ReLU here is a batch-norm output. The decomposition in section 1 needs the gradient with respect to exactly that batch-norm output, so in-place writes are switched off. The prediction does not change: only the place where the intermediate tensor is stored changes.

## 1. The FullGrad identity: where the logit goes

The lesson writes the identity for a network with biases $b$:

$$f(x; b) = \nabla_x f(x; b)^{\top} x + \nabla_b f(x; b)^{\top} b$$

It is exact for a network that is piecewise linear in its input and its biases: ReLU, max pooling, convolutions, the additions of residual connections and batch normalization in `eval()` mode. All of that describes ResNet-50.

The convolutions of ResNet-50 have no explicit biases (`bias=False`), so the biases of the network are the **implicit biases of batch normalization** plus the bias of the last linear layer. In `eval()` mode the batch normalization of channel $c$ is the affine map $\gamma_c \frac{z - \mu_c}{\sqrt{\sigma^2_c + \varepsilon}} + \beta_c$, and its bias is

$$b_c = \beta_c - \frac{\mu_c \gamma_c}{\sqrt{\sigma^2_c + \varepsilon}},$$

where $\mu_c$ and $\sigma^2_c$ are the running mean and variance accumulated during training. These are the "implicit biases, such as running averages of batch norm layers" of the paper. The contribution of one batch-norm layer to the logit is the sum over channels and positions of the gradient of the logit with respect to the layer output times $b_c$.

In [ ]:
bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]


def bn_bias(m):
    # the implicit bias of batch normalization in eval() mode
    return (m.bias - m.running_mean * m.weight / torch.sqrt(m.running_var + m.eps)).detach()


def decompose(model, x, target):
    # Terms of the FullGrad identity: the input, every batch-norm layer, the fc bias.
    outputs = []

    def keep(module, inp, out):
        out.retain_grad()
        outputs.append(out)

    hooks = [m.register_forward_hook(keep) for m in bns]
    xg = x.clone().requires_grad_(True)
    logit = model(xg)[0, target]
    logit.backward()
    for h in hooks:
        h.remove()
    input_term = (xg.grad * x).sum().item()
    bn_terms = [(o.grad * bn_bias(m)[None, :, None, None]).sum().item() for m, o in zip(bns, outputs)]
    return logit.item(), input_term, bn_terms, model.fc.bias[target].item(), xg.grad.detach()


logit, input_term, bn_terms, fc_term, grad_x = decompose(model, x, target)
print(f'batch-norm layers: {len(bns)}')
print(f'gradient times input (Gradient×Input): {input_term:9.4f}')
print(f'batch-norm biases:                     {sum(bn_terms):9.4f}')
print(f'fc bias:                               {fc_term:9.4f}')
print(f'sum of the terms:                      {input_term + sum(bn_terms) + fc_term:9.4f}')
print(f'class logit:                           {logit:9.4f}')

The sum of the terms matches the logit in all four printed decimal places: the identity holds, and nothing is thrown away.

**Task 1.** What share of the class logit does the Gradient×Input term give, that is, the ratio `input_term / logit`? What share falls on all the biases together (batch normalization and `fc`)? Round both shares to three decimals and compare them with the lesson's claim that for a network with batch normalization the part of the output that Gradient×Input throws away is "not a correction but a substantial share".

In [ ]:
# Your code here

**Your answer:**

## 2. The FullGrad map: absolute value before the sum

The map of the lesson lifts every term to the input size and adds their absolute values:

$$S_f(x) = \psi\big(\nabla_x f(x) \odot x\big) + \sum_c \psi\big(f^b(x)_c\big),$$

where $\psi$ is the absolute value, a normalization and bilinear interpolation to the input size. The `FullGrad` class of `grad-cam` is built the same way: it finds all batch-norm layers and all convolutional layers with a bias by itself, and no layer has to be chosen, so `target_layers` is empty. The bias of the last linear layer does not enter the map: it has no spatial structure.

First, let us see how much "mass" the terms carry before normalization: the sum of the absolute values of the terms against the logit itself.

In [ ]:
from pytorch_grad_cam import FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print(f'sum of absolute values of the terms: {abs(input_term) + sum(abs(t) for t in bn_terms) + abs(fc_term):8.2f}')
print(f'absolute value of the logit:         {abs(logit):8.2f}')

with FullGrad(model=model, target_layers=[]) as fullgrad:
    fullgrad_map = fullgrad(input_tensor=x, targets=[ClassifierOutputTarget(target)])[0]
    print(f'layers with a bias used by FullGrad: {len(fullgrad.target_layers)}')
print(f'FullGrad map: shape {fullgrad_map.shape}, min {fullgrad_map.min():.2f}, '
      f'max {fullgrad_map.max():.2f}, sum {fullgrad_map.sum():.1f}')

gxi_map = (grad_x * x).abs().sum(1)[0].numpy()        # absolute Gradient×Input, channels summed
vanilla_map = grad_x.abs().max(1).values[0].numpy()   # Vanilla Gradients as in the SmoothGrad notebook

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
for a, (m, t) in zip(ax, [(vanilla_map, 'Vanilla Gradients'), (gxi_map, '|Gradient×Input|'), (fullgrad_map, 'FullGrad')]):
    a.imshow(m, cmap='hot')
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()

The sum of the absolute values of the terms is an order of magnitude larger than the logit: terms of different signs cancel each other when added, and the absolute value removes that cancelling. So the sum of the FullGrad map has nothing to do with the logit, even more so after normalization into $[0, 1]$. "Nothing is thrown away" refers to the decomposition, not to the picture.

**Task 2.** How similar is the FullGrad map to the map of one of its own terms? Compute the Spearman correlation between `fullgrad_map` and `gxi_map` and between `fullgrad_map` and `vanilla_map` (flatten the maps into vectors). Round to three decimals. What do these numbers say about where the pattern of the FullGrad map comes from?

In [ ]:
# Your code here

**Your answer:**

## 3. NoiseGrad: noise in the weights and how its level is chosen

NoiseGrad multiplies **all** trained weights of the network by Gaussian noise with mean one and averages the maps of a base method over $M$ such copies:

$$W_i = \hat{W} \cdot \eta_i, \quad \eta_i \sim N(\mathbf{1},\ \sigma^2_{NG} I), \qquad E_{NG}(x) = \frac{1}{M}\sum_{i=1}^{M} E\big(x,\ f(\cdot,\ W_i)\big)$$

The base method $E$ here is Vanilla Gradients with the same channel reduction as in the notebook `PRACTICE_smoothgrad_sanity`: absolute value first, then the maximum over channels. The noise goes on all model parameters (`model.parameters()`), including the batch-norm $\gamma$ and $\beta$; the running averages are buffers, not parameters, and the noise does not touch them.

The authors suggest raising $\sigma_{NG}$ until the model's accuracy drops by about 5 %. We have no labelled ImageNet sample, so we take a cheaper substitute: six pictures from the `data` folder of the course repository and the share of cases where a noisy copy predicts the same class as the original model. Six pictures on ten copies are 60 predictions, which is a coarse measure, and its coarseness is worth keeping in mind.

In [ ]:
def vanilla_gradient(model, x, target):
    x = x.clone().requires_grad_(True)
    grad, = torch.autograd.grad(model(x)[0, target], x)
    return grad


def to_map(grad):
    return grad.abs().max(dim=1).values[0]


def noisy_model(model, sigma, gen):
    # A copy of the network with every parameter multiplied by N(1, sigma^2) noise.
    noisy = copy.deepcopy(model)
    with torch.no_grad():
        for p in noisy.parameters():
            p.mul_(1 + sigma * torch.randn(p.shape, generator=gen))
    return noisy.eval()


def noisegrad(model, x, target, sigma, M=10, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.stack([to_map(vanilla_gradient(noisy_model(model, sigma, gen), x, target)) for _ in range(M)]).mean(0)


def spearman(a, b):
    return spearmanr(np.ravel(a), np.ravel(b)).statistic


probe = torch.cat([load(n) for n in ('hog.jpg', 'cat.jpg', 'cat_and_dog.jpg', 'pig.png',
                                     'geranimo-ADUi-P4n-Jwds-unsplash.jpg', 'glen-carrie-4-Qr-Cip9gbx-Q-unsplash.jpg')])
with torch.no_grad():
    probe_classes = model(probe).argmax(1)
print('classes of the original model on six pictures:', [categories[c] for c in probe_classes])

vanilla = to_map(vanilla_gradient(model, x, target))
SIGMAS = (0.005, 0.01, 0.02, 0.05)
print(f'\n{"σ_NG":>6} {"same predictions":>18} {"Spearman w/ Vanilla":>20} {"Spearman of 2 seeds":>20}')
for sigma in SIGMAS:
    gen = torch.Generator().manual_seed(0)
    agree = 0
    for _ in range(10):
        with torch.no_grad():
            agree += int((noisy_model(model, sigma, gen)(probe).argmax(1) == probe_classes).sum())
    a, b = noisegrad(model, x, target, sigma, seed=1), noisegrad(model, x, target, sigma, seed=2)
    print(f'{sigma:6.3f} {agree:>11} of 60 {spearman(vanilla, a):20.3f} {spearman(a, b):20.3f}')

The columns of the table: how many of the 60 predictions of the noisy copies matched the prediction of the original model; the Spearman correlation of the NoiseGrad map ($M = 10$) with the Vanilla Gradients map; the Spearman correlation of two NoiseGrad maps computed with different noise seeds, that is, the stability of NoiseGrad itself.

**Task 3.** The authors' rule reads here as: at least 95 % of the predictions must match, that is, 57 of 60. What is the largest $\sigma_{NG}$ in the table that passes this rule? What is the Spearman correlation of NoiseGrad with Vanilla at it (to three decimals)? What does this mean for how much NoiseGrad changes the map at that noise level?

**Your answer:**

## 4. FusionGrad and the comparison with SmoothGrad

FusionGrad is SmoothGrad nested inside NoiseGrad: on each of $M$ noisy copies of the model the map is averaged over $N$ noisy copies of the input.

$$E_{FG}(x) = \frac{1}{N}\sum_{i=1}^{N} \frac{1}{M}\sum_{j=1}^{M} E\big(x + \xi_i,\ f(\cdot,\ W_j)\big)$$

The input noise level is the same as in the notebook `PRACTICE_smoothgrad_sanity`: 0.15 of the range of input values. The weight noise level is $\sigma_{NG} = 0.005$, the one that passed the authors' rule in section 3. The pass budget of every method except FusionGrad is 10 maps; FusionGrad uses $N = M = 5$, that is, 25 maps.

The cell below computes the four maps, the Spearman correlations between them, the stability of each (the correlation of two maps with different seeds), the number of backward passes and the time on this machine.

In [ ]:
SIGMA_NG, SIGMA_SG = 0.005, 0.15


def smoothgrad(model, x, target, n=10, seed=0):
    gen = torch.Generator().manual_seed(seed)
    s = SIGMA_SG * (x.max() - x.min())
    return torch.stack([to_map(vanilla_gradient(model, x + s * torch.randn(x.shape, generator=gen), target))
                        for _ in range(n)]).mean(0)


def fusiongrad(model, x, target, N=5, M=5, seed=0):
    gen = torch.Generator().manual_seed(seed)
    s = SIGMA_SG * (x.max() - x.min())
    maps = []
    for _ in range(M):
        noisy = noisy_model(model, SIGMA_NG, gen)
        for _ in range(N):
            maps.append(to_map(vanilla_gradient(noisy, x + s * torch.randn(x.shape, generator=gen), target)))
    return torch.stack(maps).mean(0)


methods = {
    'Vanilla':    (lambda seed: vanilla, 1),
    'SmoothGrad': (lambda seed: smoothgrad(model, x, target, seed=seed), 10),
    'NoiseGrad':  (lambda seed: noisegrad(model, x, target, SIGMA_NG, seed=seed), 10),
    'FusionGrad': (lambda seed: fusiongrad(model, x, target, seed=seed), 25),
}
maps, stability = {}, {}
print(f'{"method":12}{"passes":>10}{"seconds":>9}{"stability":>14}')
for name, (run, passes) in methods.items():
    start = time.time()
    maps[name] = run(1)
    seconds = time.time() - start
    stability[name] = spearman(maps[name], run(2))
    print(f'{name:12}{passes:>10}{seconds:9.1f}{stability[name]:14.3f}')

names = list(maps)
print('\nSpearman correlation between the maps')
print(' ' * 12 + ''.join(f'{n:>12}' for n in names))
for a in names:
    print(f'{a:12}' + ''.join(f'{spearman(maps[a], maps[b]):12.3f}' for b in names))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, n in zip(ax, names):
    a.imshow(maps[n].numpy(), cmap='hot')
    a.set_title(n)
    a.axis('off')
plt.tight_layout()
plt.show()

The stability of Vanilla is one by construction: there is no noise in it. The time depends on the machine; the number of passes does not, and it is the better way to compare the cost of the methods.

**Task 4.** Which map is FusionGrad closer to by the Spearman correlation, SmoothGrad or NoiseGrad? Explain why, relying on the result of task 3. Does the answer change if $\sigma_{NG}$ is raised to 0.05? Check it: recompute FusionGrad with `SIGMA_NG = 0.05` and compare the correlations. Remember to set `SIGMA_NG = 0.005` back for the next section.

In [ ]:
# Your code here

**Your answer:**

## 5. Where it breaks: the weight-randomization check

NoiseGrad perturbs the weights itself, so the natural question is whether it passes the weight-randomization check (Adebayo et al., 2018) from section 4 of the notebook `PRACTICE_smoothgrad_sanity`. Let us repeat the cascade: re-initialize the layers of the network from the top down and look at the Spearman correlation of the new map with the map of the original network. A map that explains the model must change noticeably when the model is broken.

In [ ]:
def randomize(model, blocks, seed=0):
    # A copy of the network with randomly re-initialized weights in the listed blocks.
    torch.manual_seed(seed)
    broken = copy.deepcopy(model)
    for name in blocks:
        for m in getattr(broken, name).modules():
            if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
                m.reset_parameters()
    return broken.eval()


cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]
print(f'{"re-initialized":32}{"Vanilla":>10}{"NoiseGrad":>11}')
for blocks in cascade:
    broken = randomize(model, blocks)
    v = spearman(vanilla, to_map(vanilla_gradient(broken, x, target)))
    n = spearman(maps['NoiseGrad'], noisegrad(broken, x, target, SIGMA_NG, seed=1))
    print(f'{", ".join(blocks):32}{v:10.3f}{n:11.3f}')

**Task 5.** Does NoiseGrad pass the check the same way Vanilla Gradients does? Compare the two columns and explain the similarity or the difference, relying on how much NoiseGrad at $\sigma_{NG} = 0.005$ differs from Vanilla.

**Your answer:**

## What to take away from this notebook

- **The FullGrad identity is exact.** For ResNet-50 the sum of the gradient times the input and the contributions of the implicit batch-norm biases matches the logit in all four printed decimal places.
- **The bias contribution can exceed the whole logit.** On this picture the gradient times the input is negative, and the whole logit, with room to spare, comes from the biases. A method that throws them away decomposes a different output.
- **The FullGrad map is not a decomposition.** The absolute value is taken before the sum, the sum of the absolute values is an order of magnitude larger than the logit, and the pattern of the map is set by the bias contributions, not by the input gradient.
- **The weight-noise level of NoiseGrad is chosen by the model, not by taste.** On ResNet-50 the rule "accuracy drops by at most 5 %" leaves so little noise that the NoiseGrad map almost coincides with Vanilla Gradients, and FusionGrad with SmoothGrad.
- **Cost is counted in passes.** SmoothGrad and NoiseGrad take $n$ or $M$ passes, FusionGrad takes $N \times M$.